<a href="https://colab.research.google.com/github/BaileyDalton007/SEEN/blob/main/paper_2/discipline_over_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

"""
Load an IDEA dataset CSV file into a dataframe with some built in preprocessing.

Args:
    file_name: file name of the csv in the raw_data folder to import
    skip_rows: number of rows to skip at the top of the csv document
    null_vals: values to replace with null
    column_mappings: dictionary that maps column names defined in the csv to the desired column names
    drop_columns: list of column names to drop

Returns:
    A pandas dataframe with the loaded IDEA data
"""
def load_IDEA(file_name, skip_rows=0, null_vals=[-8, -9], column_mappings=None, drop_columns=None):
    file_path = '/content/data/' + file_name

    # the IDEA data often has notes at the top of the CSV, drop them from the dataframe.
    df = pd.read_csv(file_path, skiprows=skip_rows)

    # replace certain values with NaNs, described in the data notes.
    df = df.replace(null_vals, np.nan)

    # drop any specified columns.
    if drop_columns:
        df = df.drop(drop_columns, axis=1)

    # if given a dictionary mapping orginal column names to new ones, rename them.
    if column_mappings:
        df = df.rename(columns=column_mappings)

    return df

In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
import math

In [3]:
#gives no information
DISCIPLINE_COLS_TO_DROP = [
    #Don't care about after expulsions
    "Children with Disability Received Educational Services during Expulsion",
    "Children with Disability Did not Receive Educational Services during Expulsion",
    "Children without Disability Received Educational Services during Expulsion",
    "Children without Disability Did not Receive Educational Services during Expulsion",
]

DISCIPLINE_COL_MAPPING = {
    'School Year' : 'Year',
    'State Name': 'State',
    'SEA Category' : 'Disability Type',
    'Children Unilaterally Removed to an IAES': 'Alternative Education(IAES)',
    'Unilateral Removals for Drugs': 'Drug-related Removal',
    'Unilateral Removals for Weapons' : 'Weapon-related Removal',
    'Unilateral Removals for Serious Bodily Injury' : 'Assault-related Removal',
    'Children Removed by Hearing Officer likely injury' : 'Removed by Officer for Likely Injury',
    'Children Suspended Expelled 10 Days or Less OSS' : '<=10 Days Suspension',
    'Children Suspended Expelled more than 10 Days OSS' : '>10 Days Suspension',
    'Children Suspended 10 Days or Less ISS': '<=10 Days Detention',
    'Children Suspended more than 10 Days ISS' : '>10 Days Detention',
    'Total Disciplinary Removals': 'Total Expulsions',
    'Children with Disciplinary Removals Totaling 1 Day': '1 Day Removals',
    'Children with Disciplinary Removals Totaling 2 to 10 Days': '2-10 Day Removals',
    'Children with Disciplinary Removals Totaling greater than 10 Days': '>10 Day Removals',
}




In [4]:
files = [
    'bdiscipline2020-21.csv',
    'bdiscipline2021-22.csv',
    'bdiscipline2022-23.csv'
]


dfs = []

for file_name in files:
    df = load_IDEA(file_name,
                skip_rows=4,
                drop_columns=DISCIPLINE_COLS_TO_DROP,
                column_mappings=DISCIPLINE_COL_MAPPING
                )

    dfs.append(df)

discipline_df = pd.concat(dfs, ignore_index=True)
discipline_df.head()

,Year,State,Disability Type,Alternative Education(IAES),Drug-related Removal,Weapon-related Removal,Assault-related Removal,Removed by Officer for Likely Injury,<=10 Days Suspension,>10 Days Suspension,<=10 Days Detention,>10 Days Detention,Total Expulsions,1 Day Removals,2-10 Day Removals,>10 Day Removals,Unnamed: 20
0,2020-21,Alabama,All Disabilities,69,51,12,11,0,4244,111,3228,183,12681,2012,4794,278,NaN
1,2020-21,Alabama,American Indian or Alaska Native,0,0,0,0,0,22,0,16,0,48,8,25,0,NaN
2,2020-21,Alabama,Asian,0,0,0,0,0,10,0,3,0,17,3,10,0,NaN
3,2020-21,Alabama,Autism,1,0,0,1,0,243,4,137,2,604,142,215,6,NaN
4,2020-21,Alabama,Black or African American,26,19,7,2,0,1986,52,1277,98,5799,774,2185,144,NaN


In [5]:


# pivot the table to combine rows for LEAs
index_cols = ['State', 'Disability Type']
values_cols = [
    'Alternative Education(IAES)',
    'Drug-related Removal',
    'Weapon-related Removal',
    'Assault-related Removal',
    'Removed by Officer for Likely Injury',
    '<=10 Days Suspension',
    '>10 Days Suspension',
    '<=10 Days Detention',
    '>10 Days Detention',
    'Total Expulsions',
    '1 Day Removals',
    '2-10 Day Removals',
    '>10 Day Removals'
]

# Convert the 'values_cols' to numeric before pivoting
for col in values_cols:
    discipline_df[col] = pd.to_numeric(discipline_df[col], errors='coerce')

df_pivot = discipline_df.pivot_table(
    index=index_cols,
    columns='Year',
    values=values_cols
).reset_index()

# some magic to rename columns reasonably.
df_pivot.columns = index_cols + [f'{col[0]} {col[1]}' for col in df_pivot.columns[len(index_cols):]]

#remove race-related
disabilities = [
    'All Disabilities',
    'Autism',
    'Deaf-blindness',
    'Developmental delay',
    'Emotional disturbance',
    'Hearing impairment',
    'Intellectual disability',
    'Multiple disabilities',
    'Orthopedic impairment',
    'Other health impairment',
    'Specific learning disability',
    'Speech or language impairment',
    'Traumatic brain injury',
]
df = df_pivot.copy()
df_disabilities = df[df["Disability Type"].isin(disabilities)]
df.head(13)






,State,Disability Type,1 Day Removals 2020-21,1 Day Removals 2021-22,1 Day Removals 2022-23,2-10 Day Removals 2020-21,2-10 Day Removals 2021-22,2-10 Day Removals 2022-23,<=10 Days Detention 2020-21,<=10 Days Detention 2021-22,...,Drug-related Removal 2022-23,Removed by Officer for Likely Injury 2020-21,Removed by Officer for Likely Injury 2021-22,Removed by Officer for Likely Injury 2022-23,Total Expulsions 2020-21,Total Expulsions 2021-22,Total Expulsions 2022-23,Weapon-related Removal 2020-21,Weapon-related Removal 2021-22,Weapon-related Removal 2022-23
0,Alabama,All Disabilities,2012.0,3792.0,4112.0,4794.0,8394.0,11782.0,3228.0,6605.0,...,210.0,0.0,0.0,0.0,12681.0,22484.0,31408.0,12.0,43.0,73.0
1,Alabama,American Indian or Alaska Native,8.0,14.0,20.0,25.0,36.0,53.0,16.0,23.0,...,1.0,0.0,0.0,0.0,48.0,69.0,120.0,0.0,0.0,1.0
2,Alabama,Asian,3.0,7.0,11.0,10.0,15.0,19.0,3.0,12.0,...,0.0,0.0,0.0,0.0,17.0,42.0,67.0,0.0,0.0,0.0
3,Alabama,Autism,142.0,225.0,347.0,215.0,417.0,673.0,137.0,313.0,...,2.0,0.0,0.0,0.0,604.0,1005.0,1739.0,0.0,1.0,3.0
4,Alabama,Black or African American,774.0,1738.0,1747.0,2185.0,4350.0,6118.0,1277.0,2981.0,...,107.0,0.0,0.0,0.0,5799.0,12262.0,17434.0,7.0,22.0,34.0
5,Alabama,Deaf-blindness,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
6,Alabama,Developmental delay,44.0,67.0,129.0,74.0,112.0,153.0,28.0,47.0,...,0.0,0.0,0.0,0.0,203.0,271.0,402.0,0.0,0.0,0.0
7,Alabama,Emotional disturbance,84.0,109.0,97.0,240.0,321.0,364.0,162.0,239.0,...,7.0,0.0,0.0,0.0,751.0,1070.0,1236.0,1.0,0.0,2.0
8,Alabama,English learner (EL) Student,64.0,115.0,NaN,121.0,263.0,NaN,101.0,237.0,...,NaN,0.0,0.0,NaN,341.0,611.0,NaN,0.0,0.0,NaN
9,Alabama,English learners,NaN,NaN,164.0,NaN,NaN,361.0,NaN,NaN,...,10.0,NaN,NaN,0.0,NaN,NaN,895.0,NaN,NaN,1.0


In [12]:
#Make our baseline, assuming every student is either male or female
#I know this is ugly as hell but I don't know Pandas
df_baseline = df[df["Disability Type"].isin(["Male", "Female"])]
df_states = []
for i in range(math.floor(len(df_baseline)/2)):
  df_state = df_baseline[2*i:2*i+2]
  state_name = df_state['State'].iloc[0]
  df_state = df_state.drop(columns=['State']).sum()
  df_state = df_state[1:] #get rid of the gender thing
  df_state = df_state.to_frame().transpose()

  #add back state
  state_col = pd.DataFrame({'State': [state_name]})
  df_state = pd.concat([state_col, df_state], axis=1)

  df_states.append(df_state)

#concat back to a single dataframe
df_baseline = pd.concat(df_states)
df_baseline.tail()

,State,1 Day Removals 2020-21,1 Day Removals 2021-22,1 Day Removals 2022-23,2-10 Day Removals 2020-21,2-10 Day Removals 2021-22,2-10 Day Removals 2022-23,<=10 Days Detention 2020-21,<=10 Days Detention 2021-22,<=10 Days Detention 2022-23,...,Drug-related Removal 2022-23,Removed by Officer for Likely Injury 2020-21,Removed by Officer for Likely Injury 2021-22,Removed by Officer for Likely Injury 2022-23,Total Expulsions 2020-21,Total Expulsions 2021-22,Total Expulsions 2022-23,Weapon-related Removal 2020-21,Weapon-related Removal 2021-22,Weapon-related Removal 2022-23
0,Virginia,1067.0,23830.0,5647.0,1709.0,27262.0,14429.0,1298.0,23075.0,13666.0,...,14.0,0.0,0.0,1.0,4453.0,53436.0,61543.0,0.0,0.0,2.0
0,Washington,523.0,3123.0,3623.0,770.0,6612.0,7961.0,390.0,4127.0,5154.0,...,112.0,0.0,0.0,10.0,2127.0,25520.0,30889.0,6.0,67.0,53.0
0,West Virginia,787.0,1611.0,1751.0,1871.0,4458.0,4763.0,1529.0,3961.0,4114.0,...,7.0,0.0,2.0,2.0,5035.0,20589.0,19373.0,1.0,0.0,1.0
0,Wisconsin,2299.0,4523.0,5427.0,2410.0,7894.0,8483.0,2010.0,4582.0,5411.0,...,0.0,1.0,0.0,0.0,8519.0,33670.0,37668.0,0.0,0.0,0.0
0,Wyoming,444.0,411.0,506.0,790.0,1052.0,1137.0,708.0,825.0,952.0,...,27.0,1.0,11.0,8.0,3031.0,4024.0,4499.0,1.0,6.0,7.0
